# Analisis Transisi Nilai G1 → G2 → G3
## Apa yang membedakan siswa *comeback* dari siswa yang tetap gagal?
**Dataset:** UCI Student Performance (Cortez & Silva, 2008). Matematika sebagai data utama, Portugis sebagai pembanding.

**Cara menjalankan di Google Colab:** pilih **Runtime > Run all**. Jika `student-mat.csv` dan `student-por.csv` belum ada, sel bagian Data akan meminta upload. Semua grafik dan tabel tersimpan di folder `hasil_analisis` dan diunduh sebagai zip di sel terakhir. Waktu jalan sekitar 1 menit.

**Peta notebook**

| Bagian | Isi | Tujuan dosen |
|---|---|---|
| Pendahuluan | Latar belakang, tantangan dan ide, tujuan penelitian, data | Struktur PPT |
| **A. EDA dasar** | Cleaning, distribusi nilai, hubungan nilai sebelumnya, waktu belajar, ketidakhadiran dengan nilai akhir, heatmap korelasi, faktor yang berkaitan dengan G3 | 1, 2, 3, 4 |
| **B. Temuan utama: comeback** | Pola transisi G1 → G2 → G3, perbandingan 30 faktor, uji ketahanan, perbandingan Matematika dan Portugis, profil risiko | 3, 4, 5 |
| Diskusi dan kesimpulan | Kesimpulan otomatis dan peta slide presentasi 20 menit | 5 |

Setiap bagian diawali catatan singkat: **apa yang dilakukan**, **mengapa**, dan **cara membaca hasilnya**.

## Latar belakang dan motivasi
Kelulusan siswa ditentukan sepanjang tahun ajaran, bukan hanya di ujian akhir. Sekolah perlu tahu **siswa mana yang masih bisa ditolong dan kapan**. Dataset UCI Student Performance mencatat nilai tiga periode (G1, G2, G3) sekaligus latar belakang keluarga, sekolah, dan gaya hidup siswa di dua sekolah menengah di Portugal, sehingga cocok untuk menelusuri perjalanan nilai siswa.

## Tantangan dan ide
**Tantangan:** G1 dan G2 hampir menentukan G3. Analisis biasa "faktor apa yang memengaruhi nilai akhir" akhirnya hanya menyimpulkan bahwa siswa yang awalnya bagus akan tetap bagus. Kesimpulan itu kurang berguna bagi sekolah.

**Ide:** fokus pada siswa yang **gagal di periode pertama (G1 < 10)**, lalu bandingkan siswa yang **bangkit dan lulus di G3 (comeback)** dengan siswa yang **tetap gagal**. Pertanyaannya bergeser dari *siapa yang pintar* menjadi *siapa yang masih bisa membalikkan keadaan*.

## Tujuan penelitian
1. Membersihkan dan menyiapkan data, termasuk menangani nilai yang tidak wajar.
2. Memahami distribusi nilai serta hubungan waktu belajar, ketidakhadiran, dan nilai sebelumnya dengan nilai akhir.
3. Memvisualisasikan pola dengan bar chart, histogram, box plot, scatter plot, dan heatmap korelasi.
4. Mengidentifikasi faktor yang paling berkaitan dengan nilai akhir.
5. Menemukan faktor yang membedakan siswa comeback dari siswa yang tetap gagal, lalu membandingkannya antara Matematika dan Portugis.

**Catatan penting:** semua temuan adalah **asosiasi, bukan sebab-akibat**, karena data berasal dari survei satu waktu.

## Setup dan konfigurasi
**Apa yang dilakukan:** memuat library dan menetapkan semua ambang keputusan.

**Mengapa:** ambang ditetapkan **sebelum** analisis agar tidak diubah setelah melihat hasil (menghindari data snooping).

In [ ]:
import os, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import plotly.graph_objects as go
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

CONFIG = {
    'AMBANG_LULUS': 10,        # lulus jika nilai >= 10 (skala 0 sampai 20, sistem Portugal)
    'ALPHA': 0.05,             # batas q-value setelah koreksi Benjamini-Hochberg
    'ALPHA_INDIKASI': 0.10,    # q antara 0,05 dan 0,10 disebut "indikasi"
    'EFEK_MIN': 0.20,          # ukuran efek minimal agar dianggap bermakna
    'EFEK_REPLIKASI': 0.10,    # efek minimal di dataset lain agar dianggap searah
    'STABILITAS_MIN': 0.70,    # proporsi bootstrap minimal agar dianggap stabil
    'N_BOOTSTRAP': 1000,
    'N_PERMUTASI': 5000,       # chi-square permutasi untuk sel kecil
    'MIN_N_KATEGORI': 10,      # kategori nominal lebih kecil dari ini digabung ke "other"
    'MIN_N_KELOMPOK': 10,      # kategori siswa lebih kecil dari ini tidak ikut uji antar kategori
    'BATAS_DOMINAN': 0.95,     # faktor dengan satu nilai > 95% dikeluarkan
    'SERTAKAN_SEMENTARA_DI_BINER': False,  # True: Comeback Sementara masuk kelompok gagal
    'SEED': 42,
}
a_lulus, A, AI, E = CONFIG['AMBANG_LULUS'], CONFIG['ALPHA'], CONFIG['ALPHA_INDIKASI'], CONFIG['EFEK_MIN']
# Struktur folder: data/ (dataset), output/gambar|tabel|interaktif (hasil)
DATA_DIR = '../data' if os.path.isdir('../data') else '.'
OUT = '../output' if os.path.isdir('../output') else 'hasil_analisis'
for _sub in ('gambar', 'tabel', 'interaktif'):
    os.makedirs(os.path.join(OUT, _sub), exist_ok=True)
RNG = np.random.default_rng(CONFIG['SEED'])

URUTAN_FOKUS = ['Comeback Awal', 'Comeback Akhir', 'Comeback Sementara', 'Tetap Gagal']
URUTAN_SEMUA = URUTAN_FOKUS + ['Stabil Lulus', 'Tergelincir']
WARNA = {'Comeback Awal': '#2E7D32', 'Comeback Akhir': '#81C784', 'Comeback Sementara': '#F9A825',
         'Tetap Gagal': '#C62828', 'Stabil Lulus': '#1565C0', 'Tergelincir': '#8E24AA',
         'Lulus': '#2E7D32', 'Gagal': '#C62828', 'Matematika': '#1565C0', 'Portugis': '#EF6C00'}
WARNA_STATUS = {'Lolos': '#2E7D32', 'Indikasi': '#F9A825', 'Tidak bermakna': '#9E9E9E'}

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
                     'axes.titleweight': 'bold', 'axes.titlesize': 11})


def simpan(fig, nama):
    fig.savefig(os.path.join(OUT, 'gambar', f'{nama}.png'))
    plt.show()
    plt.close(fig)


def simpan_tabel(df, nama):
    df.to_csv(os.path.join(OUT, 'tabel', f'{nama}.csv'), index=False)


def angka(x, d=2):
    return '-' if pd.isna(x) else f'{x:.{d}f}'.replace('.', ',')


def pct(x, d=0):
    return '-' if pd.isna(x) else f'{100 * x:.{d}f}%'.replace('.', ',')


def teks_p(nama, v):
    return f'{nama} -' if pd.isna(v) else (f'{nama} < 0,001' if v < 0.001 else f'{nama} = {angka(v, 3)}')


def daftar(xs):
    xs = list(xs)
    if not xs:
        return 'tidak ada'
    return xs[0] if len(xs) == 1 else ', '.join(xs[:-1]) + ' dan ' + xs[-1]


def label_kategori(df, urut):
    n = df.kategori.value_counts()
    return [f'{k}\n(n = {n.get(k, 0)})' for k in urut]


print('Setup selesai. Output disimpan di:', os.path.abspath(OUT))

## Data
**Apa yang dilakukan:** memuat dua file, menyusun kamus data, dan meringkas kondisi awal data.

**Mengapa empat tipe data:** uji statistik bergantung pada arti angka, bukan cara angka disimpan.
- **Biner** (13): dua pilihan, misalnya yes/no. Dikodekan 1/0.
- **Nominal** (4): lebih dari dua pilihan tanpa urutan (Mjob, Fjob, reason, guardian).
- **Ordinal** (11): skala berurutan 0 sampai 5, jarak antar nilai tidak pasti sama.
- **Numerik** (2): angka sungguhan (age, absences).

Untuk korelasi, biner, ordinal, dan numerik bisa diperlakukan sebagai angka (26 variabel), sedangkan 4 variabel nominal dianalisis terpisah.

**Variabel yang dipakai:** G3 adalah nilai akhir. G1 dan G2 adalah nilai sebelumnya: G1 dipakai sebagai kontrol, G2 sebagai checkpoint. 30 variabel latar belakang menjadi kandidat faktor.

In [ ]:
FILES = {'Matematika': os.path.join(DATA_DIR, 'student-mat.csv'),
         'Portugis': os.path.join(DATA_DIR, 'student-por.csv')}
KODE = {'Matematika': 'mat', 'Portugis': 'por'}
LAIN = {'Matematika': 'Portugis', 'Portugis': 'Matematika'}


def pastikan_file():
    hilang = [f for f in FILES.values() if not os.path.exists(f)]
    if not hilang:
        return
    try:
        from google.colab import files
        print('Silakan upload:', ', '.join(hilang))
        files.upload()
    except ImportError:
        raise FileNotFoundError(f'File tidak ditemukan: {hilang}. Letakkan di folder yang sama dengan notebook.')
    hilang = [f for f in FILES.values() if not os.path.exists(f)]
    if hilang:
        raise FileNotFoundError(f'Nama file harus persis sama: {hilang}')


def baca_csv(path):
    # pemisah koma atau titik koma dideteksi otomatis (file asli UCI memakai titik koma)
    with open(path, 'r', encoding='utf-8-sig') as fh:
        header = fh.readline()
    sep = ';' if header.count(';') > header.count(',') else ','
    df = pd.read_csv(path, sep=sep)
    df.columns = [str(c).strip().strip('"') for c in df.columns]
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip().str.strip('"')
    return df


pastikan_file()
DATA_MENTAH = {nama: baca_csv(f) for nama, f in FILES.items()}

KAMUS = pd.DataFrame([
    ('school', 'Sekolah (GP / MS)', 'biner', 'latar belakang'),
    ('sex', 'Jenis kelamin (F / M)', 'biner', 'latar belakang'),
    ('age', 'Usia (15 sampai 22)', 'numerik', 'latar belakang'),
    ('address', 'Tempat tinggal (U urban / R rural)', 'biner', 'latar belakang'),
    ('famsize', 'Ukuran keluarga (LE3 <= 3 / GT3 > 3)', 'biner', 'latar belakang'),
    ('Pstatus', 'Status orang tua (T bersama / A terpisah)', 'biner', 'latar belakang'),
    ('Medu', 'Pendidikan ibu (0 sampai 4)', 'ordinal', 'latar belakang'),
    ('Fedu', 'Pendidikan ayah (0 sampai 4)', 'ordinal', 'latar belakang'),
    ('Mjob', 'Pekerjaan ibu', 'nominal', 'latar belakang'),
    ('Fjob', 'Pekerjaan ayah', 'nominal', 'latar belakang'),
    ('reason', 'Alasan memilih sekolah', 'nominal', 'latar belakang'),
    ('guardian', 'Wali siswa', 'nominal', 'latar belakang'),
    ('traveltime', 'Waktu tempuh ke sekolah (1 sampai 4)', 'ordinal', 'latar belakang'),
    ('studytime', 'Waktu belajar per minggu (1 sampai 4)', 'ordinal', 'dapat diintervensi'),
    ('failures', 'Riwayat tidak naik kelas (0 sampai 3)', 'ordinal', 'latar belakang'),
    ('schoolsup', 'Bimbingan tambahan dari sekolah', 'biner', 'dapat diintervensi'),
    ('famsup', 'Dukungan belajar dari keluarga', 'biner', 'dapat diintervensi'),
    ('paid', 'Les berbayar mata pelajaran ini', 'biner', 'dapat diintervensi'),
    ('activities', 'Ekstrakurikuler', 'biner', 'dapat diintervensi'),
    ('nursery', 'Pernah ikut TK', 'biner', 'latar belakang'),
    ('higher', 'Ingin lanjut kuliah', 'biner', 'dapat diintervensi'),
    ('internet', 'Akses internet di rumah', 'biner', 'dapat diintervensi'),
    ('romantic', 'Sedang berpacaran', 'biner', 'dapat diintervensi'),
    ('famrel', 'Kualitas hubungan keluarga (1 sampai 5)', 'ordinal', 'latar belakang'),
    ('freetime', 'Waktu luang (1 sampai 5)', 'ordinal', 'dapat diintervensi'),
    ('goout', 'Frekuensi keluar bersama teman (1 sampai 5)', 'ordinal', 'dapat diintervensi'),
    ('Dalc', 'Konsumsi alkohol hari kerja (1 sampai 5)', 'ordinal', 'dapat diintervensi'),
    ('Walc', 'Konsumsi alkohol akhir pekan (1 sampai 5)', 'ordinal', 'dapat diintervensi'),
    ('health', 'Kondisi kesehatan (1 sampai 5)', 'ordinal', 'latar belakang'),
    ('absences', 'Jumlah ketidakhadiran', 'numerik', 'dapat diintervensi'),
    ('G1', 'Nilai periode 1 (0 sampai 20)', 'nilai', 'kontrol dan pembentuk label'),
    ('G2', 'Nilai periode 2 (0 sampai 20)', 'nilai', 'checkpoint dan pembentuk label'),
    ('G3', 'Nilai akhir (0 sampai 20)', 'nilai', 'target'),
], columns=['kolom', 'arti', 'tipe', 'peran'])

TIPE = dict(zip(KAMUS.kolom, KAMUS.tipe))
PERAN = dict(zip(KAMUS.kolom, KAMUS.peran))
FAKTOR = KAMUS.loc[KAMUS.tipe != 'nilai', 'kolom'].tolist()
NON_NOMINAL = [f for f in FAKTOR if TIPE[f] != 'nominal']
LEVEL_POSITIF = {'school': 'MS', 'sex': 'M', 'address': 'U', 'famsize': 'GT3', 'Pstatus': 'T'}
LEVEL_NEGATIF = {'school': 'GP', 'sex': 'F', 'address': 'R', 'famsize': 'LE3', 'Pstatus': 'A'}
for f in FAKTOR:
    if TIPE[f] == 'biner':
        LEVEL_POSITIF.setdefault(f, 'yes')
        LEVEL_NEGATIF.setdefault(f, 'no')
NAMA = {'school': 'sekolah', 'sex': 'jenis kelamin', 'age': 'usia', 'address': 'tempat tinggal',
        'famsize': 'ukuran keluarga', 'Pstatus': 'status tinggal orang tua', 'Medu': 'pendidikan ibu',
        'Fedu': 'pendidikan ayah', 'Mjob': 'pekerjaan ibu', 'Fjob': 'pekerjaan ayah',
        'reason': 'alasan memilih sekolah', 'guardian': 'wali siswa', 'traveltime': 'waktu tempuh',
        'studytime': 'waktu belajar', 'failures': 'riwayat tidak naik kelas',
        'schoolsup': 'bimbingan tambahan sekolah', 'famsup': 'dukungan belajar keluarga',
        'paid': 'les berbayar', 'activities': 'ekstrakurikuler', 'nursery': 'riwayat TK',
        'higher': 'keinginan kuliah', 'internet': 'akses internet', 'romantic': 'status berpacaran',
        'famrel': 'kualitas hubungan keluarga', 'freetime': 'waktu luang',
        'goout': 'frekuensi keluar bersama teman', 'Dalc': 'konsumsi alkohol hari kerja',
        'Walc': 'konsumsi alkohol akhir pekan', 'health': 'kondisi kesehatan',
        'absences': 'jumlah ketidakhadiran'}

for nama, df in DATA_MENTAH.items():
    kurang = set(KAMUS.kolom) - set(df.columns)
    assert not kurang, f'{nama}: kolom tidak ditemukan {kurang}'

KUNCI_SISWA = ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob',
               'reason', 'nursery', 'internet']
n_irisan = len(pd.merge(DATA_MENTAH['Matematika'], DATA_MENTAH['Portugis'], on=KUNCI_SISWA))

ringkas = pd.DataFrame([{'dataset': n, 'siswa': len(d), 'kolom': d.shape[1], 'missing': int(d.isna().sum().sum()),
                         'duplikat': int(d.duplicated().sum()), 'G3 = 0': int((d.G3 == 0).sum()),
                         'lulus G3': pct((d.G3 >= a_lulus).mean())} for n, d in DATA_MENTAH.items()])
display(ringkas)
print(f'Siswa yang tercatat di kedua dataset: sekitar {n_irisan}. Karena itu kedua dataset dianalisis terpisah, '
      'tidak digabung.')
print(f'Tipe faktor: ' + ', '.join(f'{t} {int((KAMUS.tipe == t).sum())}' for t in ['biner', 'nominal', 'ordinal', 'numerik']))
display(KAMUS)

# Bagian A. EDA dasar
## A1. Pembersihan data (tujuan 1)
**Apa yang dilakukan**
1. Cek missing value, duplikat, rentang angka, dan nilai kategori yang tidak sah.
2. Profilkan siswa dengan nilai 0 sebelum dibuang.
3. Keluarkan siswa dengan nilai 0 di G1, G2, atau G3.

**Mengapa nilai 0 dikeluarkan:** pada skala 0 sampai 20, nilai 0 lebih masuk akal sebagai tanda siswa tidak mengikuti penilaian (berhenti atau tidak ikut ujian), bukan hasil belajar. Jika dibiarkan, mereka otomatis masuk kelompok gagal dan mengaburkan perbandingan. Absences yang ekstrem **tidak** dibuang karena nilainya nyata, dan nanti ditangani dengan statistik berbasis peringkat.

In [ ]:
ATURAN_RENTANG = {'age': (15, 22), 'Medu': (0, 4), 'Fedu': (0, 4), 'traveltime': (1, 4), 'studytime': (1, 4),
                  'failures': (0, 3), 'famrel': (1, 5), 'freetime': (1, 5), 'goout': (1, 5), 'Dalc': (1, 5),
                  'Walc': (1, 5), 'health': (1, 5), 'absences': (0, 93), 'G1': (0, 20), 'G2': (0, 20),
                  'G3': (0, 20)}
NILAI_SAH = {'school': {'GP', 'MS'}, 'sex': {'F', 'M'}, 'address': {'U', 'R'}, 'famsize': {'LE3', 'GT3'},
             'Pstatus': {'T', 'A'}, 'Mjob': {'teacher', 'health', 'services', 'at_home', 'other'},
             'Fjob': {'teacher', 'health', 'services', 'at_home', 'other'},
             'reason': {'home', 'reputation', 'course', 'other'}, 'guardian': {'mother', 'father', 'other'}}
for f in FAKTOR:
    if TIPE[f] == 'biner' and f not in NILAI_SAH:
        NILAI_SAH[f] = {'yes', 'no'}


def profil_nilai_nol(df):
    g3 = df.G3 == 0
    return {'siswa dengan nilai 0 (G1/G2/G3)': int((df[['G1', 'G2', 'G3']] == 0).any(axis=1).sum()),
            'G3 = 0': int(g3.sum()), 'G3 = 0 dan G2 = 0': int((g3 & (df.G2 == 0)).sum()),
            'G1 = 0': int((df.G1 == 0).sum()),
            'absences = 0 pada siswa G3 = 0': pct((df.loc[g3, 'absences'] == 0).mean()) if g3.any() else '-',
            'absences = 0 pada siswa lain': pct((df.loc[~g3, 'absences'] == 0).mean())}


def bersihkan(df, nama):
    log = {'dataset': nama, 'n_awal': len(df)}
    df = df.copy()
    for c in ATURAN_RENTANG:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    kolom = KAMUS.kolom.tolist()
    log['missing'] = int(df[kolom].isna().any(axis=1).sum())
    df = df.dropna(subset=kolom)
    log['duplikat'] = int(df.duplicated().sum())
    df = df.drop_duplicates()
    luar = pd.Series(False, index=df.index)
    for c, (lo, hi) in ATURAN_RENTANG.items():
        luar |= (df[c] < lo) | (df[c] > hi)
    for c, sah in NILAI_SAH.items():
        luar |= ~df[c].isin(sah)
    log['tidak_sah'] = int(luar.sum())
    df = df[~luar]
    nol = (df[['G1', 'G2', 'G3']] == 0).any(axis=1)
    log['nilai_0_dibuang'] = int(nol.sum())
    df = df[~nol].copy()
    for c in ATURAN_RENTANG:
        df[c] = df[c].astype(int)
    log['n_akhir'] = len(df)
    log['total_dibuang'] = log['n_awal'] - log['n_akhir']
    return df.reset_index(drop=True), log


print('Profil siswa bernilai 0 sebelum dibuang:')
display(pd.DataFrame({n: profil_nilai_nol(d) for n, d in DATA_MENTAH.items()}))

DATA, LOG = {}, []
for nama, df in DATA_MENTAH.items():
    DATA[nama], log = bersihkan(df, nama)
    LOG.append(log)
LOG = pd.DataFrame(LOG)
print('Log pembersihan:')
display(LOG)
simpan_tabel(LOG, 'A1_log_cleaning')

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=False)
for ax, (nama, df) in zip(axes, DATA_MENTAH.items()):
    vc = df.G3.value_counts().reindex(range(0, 21), fill_value=0)
    warna = ['#9E9E9E' if v == 0 else (WARNA['Lulus'] if v >= a_lulus else WARNA['Gagal']) for v in vc.index]
    ax.bar(vc.index, vc.values, color=warna)
    ax.axvline(a_lulus - 0.5, ls='--', c='k', lw=1)
    ax.annotate(f'G3 = 0: {int(vc[0])} siswa', xy=(0, vc[0]), xytext=(2, vc.max() * 0.8),
                arrowprops=dict(arrowstyle='->', color='#616161'), fontsize=9)
    ax.set_title(f'{nama}: nilai akhir G3 sebelum cleaning')
    ax.set_xlabel('G3')
    ax.set_ylabel('Jumlah siswa')
fig.tight_layout()
simpan(fig, 'A1_g3_sebelum_cleaning')

## A2. Distribusi nilai (tujuan 2, histogram)
**Apa yang dilakukan:** histogram G1, G2, dan G3 setelah cleaning, dengan garis rata-rata (hitam) dan ambang lulus 10 (putus-putus merah).

**Cara membaca:** bandingkan bentuk distribusi antar periode dan seberapa banyak siswa berada tepat di sekitar ambang 10. Siswa di sekitar ambang inilah yang paling mungkin berpindah status.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey='row')
for i, (nama, df) in enumerate(DATA.items()):
    for j, g in enumerate(['G1', 'G2', 'G3']):
        ax = axes[i, j]
        ax.hist(df[g], bins=np.arange(-0.5, 21.5, 1), color=WARNA[nama], edgecolor='white', alpha=0.9)
        ax.axvline(df[g].mean(), c='k', lw=1.4)
        ax.axvline(a_lulus - 0.5, c=WARNA['Gagal'], ls='--', lw=1.4)
        ax.set_title(f'{nama}: {g} (rerata {angka(df[g].mean(), 1)}, lulus {pct((df[g] >= a_lulus).mean())})')
        ax.set_xlabel('Nilai')
        ax.set_xlim(0, 20.5)
        if j == 0:
            ax.set_ylabel('Jumlah siswa')
fig.tight_layout()
simpan(fig, 'A2_distribusi_nilai')

tabel = pd.DataFrame({n: {f'lulus {g}': pct((d[g] >= a_lulus).mean()) for g in ['G1', 'G2', 'G3']}
                      for n, d in DATA.items()})
display(tabel)

## A3. Nilai sebelumnya dan ketidakhadiran terhadap nilai akhir (tujuan 2 dan 3, scatter plot)
**Apa yang dilakukan:** scatter plot G1 vs G3, G2 vs G3, dan absences vs G3. Setiap titik adalah satu siswa, hijau jika lulus di G3 dan merah jika gagal. Garis hitam adalah garis regresi.

**Cara membaca**
- `r` adalah korelasi Pearson, `rho` adalah korelasi Spearman (berbasis peringkat, tahan terhadap nilai ekstrem).
- Nilai berupa bilangan bulat sehingga banyak titik bertumpuk. Titik digeser acak sedikit (jitter ±0,25) hanya untuk tampilan; perhitungan memakai nilai asli.
- **Kotak hijau muda di panel G1 vs G3 adalah zona comeback:** gagal di G1 tetapi lulus di G3. Siswa di zona inilah yang dianalisis di Bagian B.

In [ ]:
def jitter(x, lebar=0.25):
    return x + RNG.uniform(-lebar, lebar, len(x))


fig, axes = plt.subplots(2, 3, figsize=(16, 9.5))
for i, (nama, d) in enumerate(DATA.items()):
    lulus = (d.G3 >= a_lulus).to_numpy()
    warna = np.where(lulus, WARNA['Lulus'], WARNA['Gagal'])
    for j, (kol, label) in enumerate([('G1', 'G1 (nilai periode 1)'), ('G2', 'G2 (nilai periode 2)'),
                                      ('absences', 'Jumlah ketidakhadiran')]):
        ax = axes[i, j]
        x, y = d[kol].astype(float), d.G3.astype(float)
        ax.scatter(jitter(x), jitter(y), c=warna, s=14, alpha=0.5, edgecolor='none')
        k, b = np.polyfit(x, y, 1)
        xs = np.linspace(x.min(), x.max(), 50)
        ax.plot(xs, k * xs + b, c='k', lw=1.5)
        r, rho = stats.pearsonr(x, y)[0], stats.spearmanr(x, y)[0]
        ax.text(0.97, 0.04, f'r = {angka(r)} | rho = {angka(rho)}', transform=ax.transAxes, ha='right',
                fontweight='bold', bbox=dict(fc='white', ec='none', alpha=0.85))
        ax.axhline(a_lulus - 0.5, ls='--', c='grey', lw=1)
        if kol in ('G1', 'G2'):
            ax.axvline(a_lulus - 0.5, ls='--', c='grey', lw=1)
            ax.set_xlim(0, 20.5)
        if kol == 'G1':
            n_cb = int(((d.G1 < a_lulus) & (d.G3 >= a_lulus)).sum())
            ax.add_patch(Rectangle((0, a_lulus - 0.5), a_lulus - 0.5, 21 - a_lulus, color=WARNA['Lulus'], alpha=0.08))
            ax.text(0.5, 20, f'zona comeback\n(n = {n_cb})', fontsize=9, va='top', color=WARNA['Lulus'],
                    fontweight='bold')
        ax.set_ylim(0, 20.5)
        ax.set_xlabel(label)
        ax.set_ylabel('G3 (nilai akhir)')
        ax.set_title(f'{nama}: {kol} vs G3')
legenda = [Line2D([0], [0], marker='o', lw=0, color=WARNA['Lulus'], label='Lulus G3'),
           Line2D([0], [0], marker='o', lw=0, color=WARNA['Gagal'], label='Gagal G3')]
fig.legend(handles=legenda, loc='lower center', ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout()
simpan(fig, 'A3_scatter_nilai_sebelumnya')

## A4. Waktu belajar, riwayat tidak naik kelas, dan ketidakhadiran terhadap nilai akhir (bar chart dan box plot)
**Apa yang dilakukan**
- **Bar chart:** rata-rata G3 menurut waktu belajar dan riwayat tidak naik kelas, dengan interval kepercayaan 95% dan jumlah siswa.
- **Box plot:** sebaran G3 menurut kelompok ketidakhadiran (0, 1 sampai 5, 6 sampai 10, lebih dari 10). Kelompok dibuat karena nilai absences sangat menyebar.

**Cara membaca:** `rho` Spearman di judul menunjukkan arah dan kekuatan hubungan. Batang dengan n kecil (misalnya failures = 3) perlu dibaca hati-hati.

In [ ]:
LABEL_STUDY = {1: '<2 jam', 2: '2-5 jam', 3: '5-10 jam', 4: '>10 jam'}
BIN_ABSEN, LABEL_ABSEN = [-1, 0, 5, 10, 100], ['0', '1-5', '6-10', '>10']

fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
for i, (nama, d) in enumerate(DATA.items()):
    for j, (kol, judul) in enumerate([('studytime', 'waktu belajar per minggu'),
                                      ('failures', 'riwayat tidak naik kelas')]):
        ax = axes[i, j]
        g = d.groupby(kol).G3.agg(['mean', 'std', 'count'])
        ci = (1.96 * g['std'] / np.sqrt(g['count'])).fillna(0)
        label = [LABEL_STUDY[v] if kol == 'studytime' else str(v) for v in g.index]
        ax.bar(label, g['mean'], yerr=ci, color=WARNA[nama], capsize=4, alpha=0.9)
        for k_, (m_, n_) in enumerate(zip(g['mean'], g['count'])):
            ax.text(k_, 0.6, f'{angka(m_, 1)}\nn={n_}', ha='center', color='white', fontsize=9, fontweight='bold')
        ax.axhline(a_lulus, ls='--', c=WARNA['Gagal'], lw=1)
        rho, p = stats.spearmanr(d[kol], d.G3)
        ax.set_title(f'{nama}: rata-rata G3 menurut {judul}\nrho = {angka(rho)}, {teks_p("p", p)}')
        ax.set_ylim(0, 17)
        ax.set_ylabel('Rata-rata G3 (CI 95%)')
    ax = axes[i, 2]
    kel = pd.cut(d.absences, BIN_ABSEN, labels=LABEL_ABSEN)
    sns.boxplot(x=kel, y=d.G3, order=LABEL_ABSEN, color=WARNA[nama], showfliers=False, ax=ax)
    n_kel = kel.value_counts().reindex(LABEL_ABSEN)
    ax.set_xticks(range(len(LABEL_ABSEN)))
    ax.set_xticklabels([f'{l}\n(n={n_kel[l]})' for l in LABEL_ABSEN])
    ax.axhline(a_lulus, ls='--', c=WARNA['Gagal'], lw=1)
    rho, p = stats.spearmanr(d.absences, d.G3)
    ax.set_title(f'{nama}: sebaran G3 menurut ketidakhadiran\nrho = {angka(rho)}, {teks_p("p", p)}')
    ax.set_xlabel('Jumlah ketidakhadiran')
    ax.set_ylabel('G3')
fig.tight_layout()
simpan(fig, 'A4_studytime_failures_absences')

## A5. Heatmap korelasi (tujuan 3)
**Apa yang dilakukan:** korelasi Spearman antar semua variabel angka (ordinal, numerik, dan nilai).

**Mengapa Spearman:** sebagian besar variabel adalah skala 1 sampai 5 dan absences sangat menceng, sehingga korelasi berbasis peringkat lebih tepat daripada Pearson.

**Cara membaca:** merah = bergerak searah, biru = berlawanan arah, putih = tidak berkaitan. Perhatikan blok G1, G2, G3 yang sangat merah.

In [ ]:
kol_angka = [c for c in KAMUS.kolom if TIPE[c] in ('ordinal', 'numerik', 'nilai')]
fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
for ax, (nama, df) in zip(axes, DATA.items()):
    sns.heatmap(df[kol_angka].corr(method='spearman'), ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                annot=True, fmt='.1f', annot_kws={'size': 7}, square=True, cbar=(ax is axes[-1]))
    ax.set_title(f'Korelasi Spearman: {nama}')
fig.tight_layout()
simpan(fig, 'A5_heatmap_korelasi')

## A6. Faktor yang paling berkaitan dengan nilai akhir G3 (tujuan 4)
**Apa yang dilakukan**
- **26 faktor biner, ordinal, dan numerik:** korelasi Spearman dengan G3 (`rho`), lalu **korelasi parsial** dengan G1 dikendalikan (`rho parsial`). Korelasi parsial menjawab: *apakah faktor ini masih berkaitan dengan nilai akhir setelah titik awal siswa diketahui?*
- **4 faktor nominal:** uji Kruskal-Wallis untuk melihat apakah rata-rata G3 berbeda antar kategori.
- Semua p-value dikoreksi dengan **Benjamini-Hochberg (BH)** karena banyak uji dilakukan sekaligus.

**Cara membaca grafik:** batang gelap = `rho`, batang terang = `rho parsial`. Tanda `*` berarti signifikan setelah koreksi (q < 0,05), tanda `†` berarti tetap signifikan setelah G1 dikontrol. Jika batang terang jauh lebih pendek, hubungan faktor itu dengan G3 sebagian besar "menumpang" pada G1.

**Mengapa penting untuk Bagian B:** jika G1 menjelaskan sebagian besar G3, pertanyaan yang lebih berguna adalah *apa yang membedakan siswa yang berhasil berubah*, bukan sekadar *apa yang berkaitan dengan nilai akhir*.

In [ ]:
def kodekan_biner(df):
    X = df[FAKTOR].copy()
    for f in FAKTOR:
        if TIPE[f] == 'biner':
            X[f] = (X[f] == LEVEL_POSITIF[f]).astype(int)
    return X


def spearman_parsial(x, y, z):
    # korelasi Spearman antara x dan y setelah pengaruh z dihilangkan (regresi pada peringkat)
    rx, ry, rz = (stats.rankdata(np.asarray(v, dtype=float)) for v in (x, y, z))

    def sisa(a, b):
        k, c = np.polyfit(b, a, 1)
        return a - (k * b + c)

    r = np.corrcoef(sisa(rx, rz), sisa(ry, rz))[0, 1]
    n = len(rx)
    t = r * np.sqrt((n - 3) / max(1e-12, 1 - r ** 2))
    return r, 2 * stats.t.sf(abs(t), n - 3)


def faktor_vs_g3(d):
    X = kodekan_biner(d)
    rows = []
    for f in FAKTOR:
        if TIPE[f] == 'nominal':
            grup = [d.G3[d[f] == k] for k in d[f].unique()]
            H, p = stats.kruskal(*grup)
            rerata = d.groupby(f).G3.agg(['mean', 'size']).sort_values('mean')
            rows.append(dict(faktor=f, tipe='nominal', rho=np.nan, p=p, rho_parsial=np.nan, p_parsial=np.nan,
                             epsilon2=H / (len(d) - 1),
                             ringkas=f'terendah {rerata.index[0]} ({angka(rerata["mean"].iloc[0], 1)}, '
                                     f'n={rerata["size"].iloc[0]}), tertinggi {rerata.index[-1]} '
                                     f'({angka(rerata["mean"].iloc[-1], 1)}, n={rerata["size"].iloc[-1]})'))
            continue
        rho, p = stats.spearmanr(X[f], d.G3)
        rp, pp = spearman_parsial(X[f], d.G3, d.G1)
        rows.append(dict(faktor=f, tipe=TIPE[f], rho=rho, p=p, rho_parsial=rp, p_parsial=pp, epsilon2=np.nan,
                         ringkas=''))
    T = pd.DataFrame(rows)
    T['q'] = multipletests(T.p, method='fdr_bh')[1]
    m = T.p_parsial.notna()
    T['q_parsial'] = np.nan
    T.loc[m, 'q_parsial'] = multipletests(T.loc[m, 'p_parsial'], method='fdr_bh')[1]
    T['abs_rho'] = T.rho.abs()
    return T


HASIL_A = {}
for nama, d in DATA.items():
    T = faktor_vs_g3(d)
    HASIL_A[nama] = {'faktor_g3': T, 'r_g1': stats.pearsonr(d.G1, d.G3)[0], 'r_g2': stats.pearsonr(d.G2, d.G3)[0]}
    simpan_tabel(T, f'A6_faktor_vs_g3_{KODE[nama]}')
    print(f'\n=== {nama}: 10 faktor dengan |rho| terbesar terhadap G3 ===')
    display(T[T.tipe != 'nominal'].nlargest(10, 'abs_rho')[['faktor', 'tipe', 'rho', 'q', 'rho_parsial', 'q_parsial']]
            .round(3))
    print(f'{nama}: faktor nominal (Kruskal-Wallis terhadap G3)')
    display(T[T.tipe == 'nominal'][['faktor', 'p', 'q', 'epsilon2', 'ringkas']].round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), sharex=True)
for ax, (nama, H) in zip(axes, HASIL_A.items()):
    T = H['faktor_g3']
    T = T[T.tipe != 'nominal'].nlargest(12, 'abs_rho').iloc[::-1]
    y = np.arange(len(T))
    ax.barh(y + 0.2, T.rho, height=0.38, color=WARNA[nama], label='rho (tanpa kontrol)')
    ax.barh(y - 0.2, T.rho_parsial, height=0.38, color=WARNA[nama], alpha=0.35, label='rho parsial (G1 dikontrol)')
    label = [f'{f}{" *" if q < A else ""}{" †" if qp < A else ""}' for f, q, qp in zip(T.faktor, T.q, T.q_parsial)]
    ax.set_yticks(y)
    ax.set_yticklabels(label)
    ax.axvline(0, c='k', lw=1)
    ax.set_title(f'{nama}: 12 faktor terkuat terhadap G3\n(* signifikan, † tetap signifikan setelah G1 dikontrol)')
    ax.set_xlabel('Korelasi Spearman dengan G3')
    ax.legend(loc='lower right', fontsize=8)
fig.tight_layout()
simpan(fig, 'A6_faktor_vs_g3')

### Ringkasan Bagian A
Sel berikut merangkum temuan EDA dasar secara otomatis dan menjadi jembatan ke Bagian B.

In [ ]:
for nama, H in HASIL_A.items():
    T = H['faktor_g3']
    num = T[T.tipe != 'nominal']
    top = num.nlargest(3, 'abs_rho')
    bertahan = num[num.q_parsial < A].sort_values('q_parsial').faktor.tolist()
    nom = T[(T.tipe == 'nominal') & (T.q < A)].faktor.tolist()
    H['top_g3'], H['bertahan'], H['nominal_sig'] = top, bertahan, nom
    print(f'- {nama}: korelasi G1-G3 = {angka(H["r_g1"])}, G2-G3 = {angka(H["r_g2"])}. Nilai sebelumnya adalah '
          'penentu terkuat nilai akhir.')
    print(f'  Faktor latar belakang terkuat terhadap G3: '
          + ', '.join(f'{r.faktor} (rho {angka(r.rho)})' for _, r in top.iterrows()) + '.')
    print(f'  Setelah G1 dikontrol, faktor yang tetap bermakna: {daftar(bertahan)}.')
    print(f'  Faktor nominal yang berkaitan dengan G3: {daftar(nom)}.')
print('\nKarena G1 sangat menentukan G3, Bagian B berfokus pada siswa yang gagal di G1 dan mencari apa yang '
      'membedakan siswa yang berhasil bangkit.')

# Bagian B. Temuan utama: siswa comeback
## B1. Definisi kategori (tujuan 5)
**Apa yang dilakukan:** memberi label lintasan nilai pada setiap siswa berdasarkan status lulus (nilai ≥ 10) di tiap periode.

| Kategori | G1 | G2 | G3 | Keterangan |
|---|---|---|---|---|
| Comeback Awal | gagal | lulus | lulus | Bangkit sejak periode 2 |
| Comeback Akhir | gagal | gagal | lulus | Bangkit di periode akhir |
| Comeback Sementara | gagal | lulus | gagal | Sempat bangkit, lalu gagal lagi |
| Tetap Gagal | gagal | gagal | gagal | Tidak pernah lulus |

Siswa dengan G1 ≥ 10 diberi label Stabil Lulus atau Tergelincir hanya untuk gambaran keseluruhan.

**Analisis utama:** Comeback (Awal + Akhir, lulus di G3) vs Tetap Gagal. Comeback Sementara tetap tampil di grafik, tetapi tidak dimasukkan ke perbandingan utama karena lintasannya berbeda dari kedua kelompok.

In [ ]:
def beri_label(df):
    df = df.copy()
    for g in ['G1', 'G2', 'G3']:
        df[f'lulus_{g}'] = df[g] >= a_lulus
    l1, l2, l3 = df.lulus_G1, df.lulus_G2, df.lulus_G3
    kondisi = [~l1 & l2 & l3, ~l1 & ~l2 & l3, ~l1 & l2 & ~l3, ~l1 & ~l2 & ~l3, l1 & l3, l1 & ~l3]
    df['kategori'] = np.select(kondisi, URUTAN_SEMUA, default='?')
    df['selisih_G3_G1'] = df.G3 - df.G1
    return df


HASIL = {}
rows = []
for nama, df in DATA.items():
    d = beri_label(df)
    fokus = d[~d.lulus_G1].copy()
    kel_gagal = ['Tetap Gagal'] + (['Comeback Sementara'] if CONFIG['SERTAKAN_SEMENTARA_DI_BINER'] else [])
    biner = fokus[fokus.kategori.isin(['Comeback Awal', 'Comeback Akhir'] + kel_gagal)].copy()
    biner['comeback'] = biner.kategori.isin(['Comeback Awal', 'Comeback Akhir']).astype(int)
    HASIL[nama] = {'df': d, 'fokus': fokus, 'biner': biner}
    vc = fokus.kategori.value_counts().reindex(URUTAN_FOKUS, fill_value=0)
    for k, v in vc.items():
        rows.append({'dataset': nama, 'kategori': k, 'n': int(v), 'persen': v / len(fokus)})
KOMPOSISI = pd.DataFrame(rows)
simpan_tabel(KOMPOSISI, 'B1_komposisi_kategori')
tampil = KOMPOSISI.assign(persen=KOMPOSISI.persen.map(lambda v: pct(v, 1)))
print('Kategori siswa dengan G1 < 10:')
display(tampil.pivot(index='kategori', columns='dataset', values=['n', 'persen']).reindex(URUTAN_FOKUS))
for nama, H in HASIL.items():
    b = H['biner']
    print(f'{nama}: {len(H["fokus"])} siswa gagal di G1 -> analisis utama {int(b.comeback.sum())} Comeback vs '
          f'{int((b.comeback == 0).sum())} Tetap Gagal')

## B2. Pola transisi nilai
**Apa yang dilakukan**
- **Sankey diagram (interaktif):** aliran siswa antar status lulus/gagal dari G1 ke G2 ke G3. Lebar pita = jumlah siswa. File HTML tersimpan untuk ditampilkan atau di-screenshot ke PPT.
- **Komposisi kategori:** proporsi empat kategori pada siswa G1 < 10 di kedua dataset.
- **Lintasan rata-rata:** rata-rata G1, G2, G3 per kategori, untuk melihat kapan setiap kelompok mulai naik.

In [ ]:
def gambar_sankey(df, nama):
    label = ['G1 Lulus', 'G1 Gagal', 'G2 Lulus', 'G2 Gagal', 'G3 Lulus', 'G3 Gagal']
    idx = {l: i for i, l in enumerate(label)}
    status = lambda s: np.where(s, 'Lulus', 'Gagal')
    src, tgt, val, col = [], [], [], []
    for a_, b_ in [('G1', 'G2'), ('G2', 'G3')]:
        t = pd.crosstab(status(df[f'lulus_{a_}']), status(df[f'lulus_{b_}']))
        for s_ in t.index:
            for d_ in t.columns:
                if t.loc[s_, d_] > 0:
                    src.append(idx[f'{a_} {s_}'])
                    tgt.append(idx[f'{b_} {d_}'])
                    val.append(int(t.loc[s_, d_]))
                    col.append('rgba(46,125,50,0.35)' if d_ == 'Lulus' else 'rgba(198,40,40,0.35)')
    n_node = []
    for g in ['G1', 'G2', 'G3']:
        n_node += [int(df[f'lulus_{g}'].sum()), int((~df[f'lulus_{g}']).sum())]
    fig = go.Figure(go.Sankey(
        arrangement='snap',
        node=dict(label=[f'{l} ({n})' for l, n in zip(label, n_node)], pad=22, thickness=18,
                  color=['#2E7D32', '#C62828'] * 3),
        link=dict(source=src, target=tgt, value=val, color=col)))
    fig.update_layout(title=f'Aliran status G1 → G2 → G3: {nama} (n = {len(df)})', font_size=12, height=400)
    fig.write_html(os.path.join(OUT, 'interaktif', f'B2_sankey_{KODE[nama]}.html'), include_plotlyjs='cdn')
    fig.show()


for nama, H in HASIL.items():
    gambar_sankey(H['df'], nama)

fig, axes = plt.subplots(1, 3, figsize=(19, 4.8), gridspec_kw={'width_ratios': [1.5, 1, 1]})
ax = axes[0]
komp = KOMPOSISI.pivot(index='dataset', columns='kategori', values='n')[URUTAN_FOKUS]
prop = komp.div(komp.sum(1), axis=0)
kiri = np.zeros(len(prop))
for k in URUTAN_FOKUS:
    ax.barh(prop.index, prop[k], left=kiri, color=WARNA[k], label=k, edgecolor='white')
    for i, (v, l) in enumerate(zip(prop[k], kiri)):
        if v >= 0.05:
            ax.text(l + v / 2, i, f'{pct(v)}\n({komp[k].iloc[i]})', ha='center', va='center', fontsize=9,
                    color='black' if k in ('Comeback Akhir', 'Comeback Sementara') else 'white')
    kiri += prop[k].values
ax.set_xlim(0, 1)
ax.set_title('Komposisi kategori siswa dengan G1 < 10')
ax.legend(ncol=2, loc='upper center', bbox_to_anchor=(0.5, -0.1), frameon=False, fontsize=9)

for ax, (nama, H) in zip(axes[1:], HASIL.items()):
    f_ = H['fokus']
    ada = [k for k in URUTAN_FOKUS if (f_.kategori == k).any()]
    rata = f_.groupby('kategori')[['G1', 'G2', 'G3']].mean().reindex(ada)
    n = f_.kategori.value_counts()
    for k, r in rata.iterrows():
        ax.plot(['G1', 'G2', 'G3'], r.values, marker='o', lw=2.5, color=WARNA[k], label=f'{k} (n = {n[k]})')
    ax.axhline(a_lulus, ls='--', c='k', lw=1)
    ax.set_title(f'{nama}: lintasan rata-rata nilai')
    ax.set_ylabel('Nilai rata-rata')
    ax.legend(fontsize=8, loc='upper left')
fig.tight_layout()
simpan(fig, 'B2_komposisi_dan_lintasan')

## B3. G2 sebagai checkpoint
**Apa yang dilakukan:** menghitung peluang lulus di G3 bagi siswa yang gagal di G1, dibedakan menurut status di G2, dengan interval kepercayaan Wilson 95%.

**Mengapa terpisah dari faktor:** G2 ikut membentuk kategori, sehingga tidak boleh diuji sebagai faktor (menghindari data leakage). G2 dipakai sebagai titik pemeriksaan kedua bagi sekolah.

In [ ]:
rows = []
for nama, H in HASIL.items():
    f_ = H['fokus']
    tab = []
    for lulus_g2 in [True, False]:
        sub = f_[f_.lulus_G2 == lulus_g2]
        k, n = int(sub.lulus_G3.sum()), len(sub)
        lo, hi = proportion_confint(k, n, method='wilson') if n else (np.nan, np.nan)
        rows.append({'dataset': nama, 'status G2': 'Lulus G2' if lulus_g2 else 'Gagal G2', 'n': n, 'lulus G3': k,
                     'peluang lulus G3': k / n if n else np.nan, 'CI bawah': lo, 'CI atas': hi})
        tab.append([k, n - k])
    H['g2_p'] = stats.fisher_exact(tab)[1]
G2 = pd.DataFrame(rows)
for nama, H in HASIL.items():
    H['g2'] = G2[G2.dataset == nama].set_index('status G2')
display(G2.round(3))
simpan_tabel(G2, 'B3_checkpoint_g2')

fig, ax = plt.subplots(figsize=(8, 4.3))
lebar = 0.36
for i, nama in enumerate(HASIL):
    d = G2[G2.dataset == nama]
    xpos = np.arange(2) + (i - 0.5) * lebar
    p_ = d['peluang lulus G3'].to_numpy()
    ax.bar(xpos, p_, width=lebar, color=WARNA[nama], label=nama,
           yerr=[np.clip(p_ - d['CI bawah'], 0, None), np.clip(d['CI atas'] - p_, 0, None)], capsize=4)
    for x_, v, n in zip(xpos, p_, d['n']):
        ax.text(x_, v + 0.07, f'{pct(v)}\nn={n}', ha='center', fontsize=9)
ax.set_xticks(np.arange(2))
ax.set_xticklabels(['Sudah lulus di G2', 'Masih gagal di G2'])
ax.set_ylim(0, 1.18)
ax.set_ylabel('Peluang lulus di G3')
ax.set_title('Checkpoint G2: peluang lulus G3 bagi siswa yang gagal di G1')
ax.legend()
simpan(fig, 'B3_checkpoint_g2')

## B4. Membandingkan 30 faktor: Comeback vs Tetap Gagal
**Apa yang dilakukan:** setiap faktor diuji dengan metode yang sesuai tipenya, lalu diberi ukuran efek agar bisa dibandingkan.

| Tipe faktor | Uji (apakah berbeda) | Ukuran efek (seberapa berbeda) |
|---|---|---|
| Ordinal dan numerik | Mann-Whitney U | Rank-biserial r, bertanda (setara Cliff's delta) |
| Biner | Fisher exact | Phi, bertanda |
| Nominal | Chi-square, atau permutasi jika sel kecil | Cramér's V terkoreksi bias (tanpa arah) |

Tanda positif berarti nilai faktor lebih tinggi atau lebih sering pada siswa Comeback. Patokan efek: sekitar 0,1 kecil, 0,3 sedang, 0,5 besar.

**Koreksi uji berganda:** 30 uji sekaligus membuat sebagian hasil tampak signifikan karena kebetulan. Keputusan memakai **Benjamini-Hochberg** karena analisis ini eksploratif. Kolom **Holm** (lebih ketat) ikut ditampilkan sebagai pembanding; faktor yang lolos BH tetapi tidak lolos Holm ditandai "sensitif terhadap metode koreksi".

**Status:** Lolos jika q < 0,05 dan |efek| ≥ 0,20. Indikasi jika q < 0,10 dan |efek| ≥ 0,20.

**Persiapan faktor:** kategori nominal dengan kurang dari 10 siswa digabung ke `other`, dan faktor yang satu nilainya mendominasi lebih dari 95% dikeluarkan. Keduanya hanya berdasarkan frekuensi, tidak melihat hasil comeback. Ketidakpastian ukuran efek dihitung dengan **bootstrap** (resampling 1000 kali per kelompok).

In [ ]:
def siapkan_faktor(fokus):
    X = pd.DataFrame(index=fokus.index)
    aktif, catatan = [], []
    for f in FAKTOR:
        s = fokus[f].copy()
        if TIPE[f] == 'nominal':
            vc = s.value_counts()
            jarang = vc[vc < CONFIG['MIN_N_KATEGORI']].index.tolist()
            if jarang:
                s = s.where(~s.isin(jarang), 'other')
                catatan.append(f'{f}: kategori {jarang} digabung ke "other"')
        dominan = s.value_counts(normalize=True).iloc[0]
        if s.nunique() < 2 or dominan > CONFIG['BATAS_DOMINAN']:
            catatan.append(f'{f}: dikeluarkan karena satu nilai mendominasi ({pct(dominan)})')
            continue
        X[f] = (s == LEVEL_POSITIF[f]).astype(int) if TIPE[f] == 'biner' else s
        aktif.append(f)
    return X, aktif, catatan


def chi2_stat(t):
    t = np.asarray(t, dtype=float)
    e = t.sum(1, keepdims=True) * t.sum(0, keepdims=True) / t.sum()
    with np.errstate(divide='ignore', invalid='ignore'):
        return float(np.where(e > 0, (t - e) ** 2 / e, 0.0).sum())


def cramers_v_koreksi(chi2, n, r, k):
    phi2 = max(0.0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rk, kk = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    d = min(rk - 1, kk - 1)
    return float(np.sqrt(phi2 / d)) if d > 0 else np.nan


def kode_kategori(x):
    return np.unique(np.asarray(x).astype(str), return_inverse=True)[1].ravel()


def hitung_efek(x, y, tipe):
    x, y = np.asarray(x), np.asarray(y).astype(int)
    n1 = int(y.sum())
    n0 = len(y) - n1
    if n1 < 2 or n0 < 2:
        return np.nan
    if tipe in ('ordinal', 'numerik'):
        rk = stats.rankdata(x.astype(float))
        return 2 * (rk[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0) - 1
    if tipe == 'biner':
        x = x.astype(float)
        return np.nan if x.std() == 0 else float(np.corrcoef(x, y)[0, 1])
    kode = kode_kategori(x)
    k = kode.max() + 1
    t = np.bincount(kode * 2 + y, minlength=2 * k).reshape(k, 2)
    return cramers_v_koreksi(chi2_stat(t), len(y), k, 2) if k > 1 else np.nan


def uji_kontingensi(x, g):
    xk, gk = kode_kategori(x), kode_kategori(g)
    kx, kg = xk.max() + 1, gk.max() + 1
    t = np.bincount(xk * kg + gk, minlength=kx * kg).reshape(kx, kg)
    chi2 = chi2_stat(t)
    e = t.sum(1, keepdims=True) * t.sum(0, keepdims=True) / t.sum()
    if (e < 5).any():
        hit = sum(chi2_stat(np.bincount(xk * kg + RNG.permutation(gk), minlength=kx * kg).reshape(kx, kg))
                  >= chi2 - 1e-9 for _ in range(CONFIG['N_PERMUTASI']))
        return chi2, (hit + 1) / (CONFIG['N_PERMUTASI'] + 1), 'Chi-square permutasi'
    return chi2, float(stats.chi2.sf(chi2, (kx - 1) * (kg - 1))), 'Chi-square'


def uji_semua_faktor(X, y, aktif):
    y = pd.Series(np.asarray(y).astype(int), index=X.index)
    baris = []
    for f in aktif:
        tipe, x = TIPE[f], X[f]
        x1, x0 = x[y == 1], x[y == 0]
        if tipe in ('ordinal', 'numerik'):
            p = stats.mannwhitneyu(x1.astype(float), x0.astype(float), alternative='two-sided')[1]
            uji, ringkas = 'Mann-Whitney U', f'median {x1.median():g} vs {x0.median():g}; rerata {x1.mean():.2f} vs {x0.mean():.2f}'
        elif tipe == 'biner':
            tab = [[int(((x == 1) & (y == 1)).sum()), int(((x == 1) & (y == 0)).sum())],
                   [int(((x == 0) & (y == 1)).sum()), int(((x == 0) & (y == 0)).sum())]]
            p = stats.fisher_exact(tab)[1]
            uji = 'Fisher exact'
            ringkas = (f'tingkat comeback {f} = {LEVEL_POSITIF[f]}: {pct(y[x == 1].mean())} '
                       f'vs {LEVEL_NEGATIF[f]}: {pct(y[x == 0].mean())}')
        else:
            _, p, uji = uji_kontingensi(x, y)
            laju = y.groupby(x).mean().sort_values(ascending=False)
            ringkas = 'tingkat comeback: ' + ', '.join(f'{k} {pct(v)}' for k, v in laju.items())
        baris.append(dict(faktor=f, tipe=tipe, peran=PERAN[f], uji=uji, p=p, efek=hitung_efek(x, y, tipe),
                          ringkas=ringkas))
    T = pd.DataFrame(baris)
    T['q'] = multipletests(T.p, method='fdr_bh')[1]
    T['p_holm'] = multipletests(T.p, method='holm')[1]
    T['efek_abs'] = T.efek.abs()
    T['status'] = np.select([(T.q < A) & (T.efek_abs >= E), (T.q < AI) & (T.efek_abs >= E)],
                            ['Lolos', 'Indikasi'], 'Tidak bermakna')
    T['catatan'] = np.where((T.status == 'Lolos') & (T.p_holm >= A), 'sensitif terhadap metode koreksi', '')
    return T.sort_values(['q', 'efek_abs'], ascending=[True, False]).reset_index(drop=True)


def bootstrap_efek(X, y, aktif, efek_penuh, B):
    y = np.asarray(y).astype(int)
    i1, i0 = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    arr = {f: (kode_kategori(X[f]) if TIPE[f] == 'nominal' else X[f].to_numpy()) for f in aktif}
    hasil = np.full((B, len(aktif)), np.nan)
    for b in range(B):
        s = np.concatenate([RNG.choice(i1, len(i1)), RNG.choice(i0, len(i0))])
        for j, f in enumerate(aktif):
            hasil[b, j] = hitung_efek(arr[f][s], y[s], TIPE[f])
    rows = []
    for j, f in enumerate(aktif):
        e = hasil[:, j][~np.isnan(hasil[:, j])]
        ef = efek_penuh[f]
        stabil = np.mean(e >= E) if TIPE[f] == 'nominal' else np.mean((np.sign(e) == np.sign(ef)) & (np.abs(e) >= E))
        rows.append(dict(faktor=f, CI95_bawah=np.percentile(e, 2.5), CI95_atas=np.percentile(e, 97.5),
                         stabilitas=stabil))
    return pd.DataFrame(rows)


for nama, H in HASIL.items():
    X, aktif, catatan = siapkan_faktor(H['fokus'])
    H['X_fokus'], H['aktif'] = X, aktif
    H['X'], H['y'] = X.loc[H['biner'].index], H['biner']['comeback']
    T = uji_semua_faktor(H['X'], H['y'], aktif)
    H['boot'] = bootstrap_efek(H['X'], H['y'], aktif, T.set_index('faktor').efek, CONFIG['N_BOOTSTRAP'])
    H['uji'] = T.merge(H['boot'], on='faktor')
    simpan_tabel(H['uji'], f'B4_uji_faktor_{KODE[nama]}')
    print(f'\n=== {nama}: {len(aktif)} faktor diuji, {(T.status == "Lolos").sum()} lolos, '
          f'{(T.status == "Indikasi").sum()} indikasi ===')
    for c in catatan:
        print('   catatan:', c)
    display(H['uji'][['faktor', 'tipe', 'uji', 'efek', 'CI95_bawah', 'CI95_atas', 'q', 'p_holm', 'status',
                      'catatan', 'ringkas']].head(10).round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 10))
for ax, (nama, H) in zip(axes, HASIL.items()):
    T = H['uji'].sort_values('efek').reset_index(drop=True)
    ypos = np.arange(len(T))
    warna = T.status.map(WARNA_STATUS)
    ax.hlines(ypos, T.CI95_bawah, T.CI95_atas, color=warna, lw=2.2)
    for yy, e, c, t in zip(ypos, T.efek, warna, T.tipe):
        ax.plot(e, yy, 's' if t == 'nominal' else 'o', color=c, ms=7)
    ax.set_yticks(ypos)
    ax.set_yticklabels(T.faktor)
    for v in (-E, E):
        ax.axvline(v, ls='--', c='grey', lw=0.9)
    ax.axvline(0, c='k', lw=1)
    ax.set_title(f'{nama}: ukuran efek Comeback vs Tetap Gagal (CI bootstrap 95%)')
    ax.set_xlabel('Efek (+ = lebih tinggi / lebih sering pada Comeback)')
legenda = [Line2D([0], [0], color=c, marker='o', lw=2, label=s) for s, c in WARNA_STATUS.items()]
legenda.append(Line2D([0], [0], color='k', marker='s', lw=0, label="kotak = Cramer's V (tanpa arah)"))
fig.legend(handles=legenda, loc='lower center', ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.03))
fig.tight_layout()
simpan(fig, 'B4_forest_plot_faktor')

## B5. Seberapa kuat temuannya? Uji ketahanan dan matriks konsistensi
**Masalah:** siswa Comeback cenderung punya G1 lebih tinggi (lebih dekat ke 10). Faktor yang tampak berbeda bisa saja hanya "bawaan" titik awal.

**Apa yang dilakukan:** setiap faktor dinilai pada lima pemeriksaan, lalu dijumlahkan menjadi **skor konsistensi 0 sampai 5**.

| Pemeriksaan | Cara | Lolos jika |
|---|---|---|
| 1. Signifikan | Koreksi BH (B4) | q < 0,05 |
| 2. Efek cukup besar | Ukuran efek (B4) | \|efek\| ≥ 0,20 |
| 3. Tahan kontrol G1 | Regresi logistik `comeback ~ G1 + faktor`, uji likelihood ratio, koreksi BH | q < 0,05 dan arah sama |
| 4. Stabil | Bootstrap (B4) | ≥ 70% resample searah dengan efek ≥ 0,20 |
| 5. Searah di dataset lain | Ukuran efek di Portugis (atau Matematika) | Arah sama dan \|efek\| ≥ 0,10 |

**Cara membaca:** skor 5 berarti temuan konsisten di semua pemeriksaan. Skor rendah berarti temuan rapuh atau tidak ada.

In [ ]:
def kodekan_logit(X, f):
    if TIPE[f] == 'nominal':
        urutan = X[f].value_counts().index.tolist()
        D = pd.get_dummies(pd.Categorical(X[f], categories=urutan), prefix=f, drop_first=True, dtype=float)
        D.index = X.index
        return D
    if TIPE[f] == 'numerik':
        return ((X[f] - X[f].mean()) / X[f].std()).astype(float).to_frame(f)
    return X[f].astype(float).to_frame(f)


def logit_kontrol_g1(X, y, g1, aktif):
    y = np.asarray(y).astype(int)
    dasar = pd.DataFrame({'const': 1.0, 'G1': np.asarray(g1, dtype=float)}, index=X.index)
    m0 = sm.Logit(y, dasar).fit(disp=0)
    rows = []
    for f in aktif:
        D = kodekan_logit(X, f)
        rec = dict(faktor=f, OR_kontrol_G1=np.nan, p_LR=np.nan)
        try:
            m1 = sm.Logit(y, pd.concat([dasar, D], axis=1)).fit(disp=0, maxiter=200)
            rec['p_LR'] = float(stats.chi2.sf(max(0.0, 2 * (m1.llf - m0.llf)), D.shape[1]))
            if D.shape[1] == 1:
                rec['OR_kontrol_G1'] = float(np.exp(m1.params[f]))
        except Exception:
            pass
        rows.append(rec)
    T = pd.DataFrame(rows)
    ok = T.p_LR.notna()
    T['q_LR'] = np.nan
    T.loc[ok, 'q_LR'] = multipletests(T.loc[ok, 'p_LR'], method='fdr_bh')[1]
    return T


LAPIS = ['Signifikan', 'Efek >= 0,20', 'Tahan kontrol G1', 'Stabil bootstrap', 'Searah di dataset lain']

for nama, H in HASIL.items():
    H['logit'] = logit_kontrol_g1(H['X'], H['y'], H['biner'].G1, H['aktif'])

for nama, H in HASIL.items():
    T = H['uji'].merge(H['logit'], on='faktor').set_index('faktor')
    eo = HASIL[LAIN[nama]]['uji'].set_index('faktor').efek.reindex(T.index).to_numpy()
    bertanda = (T.tipe != 'nominal').to_numpy()
    tanda = np.sign(T.efek.to_numpy())
    r_min = CONFIG['EFEK_REPLIKASI']
    M = pd.DataFrame(index=T.index)
    M[LAPIS[0]] = (T.q < A).to_numpy()
    M[LAPIS[1]] = (T.efek_abs >= E).to_numpy()
    arah_or = np.where(bertanda, np.sign(np.log(T.OR_kontrol_G1.to_numpy())) == tanda, True)
    M[LAPIS[2]] = (T.q_LR.to_numpy() < A) & arah_or
    M[LAPIS[3]] = (T.stabilitas.to_numpy() >= CONFIG['STABILITAS_MIN'])
    M[LAPIS[4]] = np.where(bertanda, (np.sign(eo) == tanda) & (np.abs(eo) >= r_min), eo >= r_min)
    M = M.fillna(False).astype(bool)
    T['skor'] = M.sum(axis=1).astype(int)
    H['konsistensi'] = M
    H['ringkas'] = T.reset_index().sort_values(['skor', 'efek_abs'], ascending=False).reset_index(drop=True)
    simpan_tabel(H['ringkas'], f'B5_ringkasan_faktor_{KODE[nama]}')

    lolos = H['ringkas'][(H['ringkas'].status == 'Lolos') & (H['ringkas'].tipe != 'nominal')].faktor.tolist()
    if len(lolos) >= 2:
        C = H['X'][lolos].astype(float).corr(method='spearman')
        H['pasangan_kuat'] = [(a_, b_, C.loc[a_, b_]) for i, a_ in enumerate(lolos) for b_ in lolos[i + 1:]
                              if abs(C.loc[a_, b_]) >= 0.5]
    else:
        H['pasangan_kuat'] = []
    print(f'\n=== {nama}: faktor dengan skor konsistensi tertinggi ===')
    display(H['ringkas'][['faktor', 'peran', 'efek', 'q', 'OR_kontrol_G1', 'q_LR', 'stabilitas', 'skor', 'status',
                          'catatan']].head(8).round(3))
    print('Pasangan faktor kunci yang berkorelasi kuat (|rho| >= 0,5):',
          [(a_, b_, round(v, 2)) for a_, b_, v in H['pasangan_kuat']] or 'tidak ada')

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax, (nama, H) in zip(axes, HASIL.items()):
    R = H['ringkas']
    R = R[R.skor >= 1].head(15).set_index('faktor')
    Mv = H['konsistensi'].loc[R.index]
    sns.heatmap(Mv.astype(int), ax=ax, cmap=ListedColormap(['#EFEFEF', '#2E7D32']), vmin=0, vmax=1, cbar=False,
                linewidths=0.8, linecolor='white', annot=np.where(Mv.values, '✓', ''), fmt='',
                annot_kws={'color': 'white', 'size': 12, 'weight': 'bold'})
    ax.set_yticks(np.arange(len(R)) + 0.5)
    ax.set_yticklabels([f'{f}  ({R.loc[f, "skor"]}/5)' for f in R.index], rotation=0)
    ax.set_xticklabels(Mv.columns, rotation=25, ha='right')
    ax.set_title(f'Matriks konsistensi: {nama}\n(faktor dengan skor minimal 1)')
    ax.set_ylabel('')
fig.tight_layout()
simpan(fig, 'B5_matriks_konsistensi')

## B6. Profil faktor kunci
**Apa yang dilakukan**
- **Bar chart tingkat comeback** per level untuk tiga faktor dengan skor konsistensi tertinggi di Matematika, ditampilkan juga untuk Portugis agar bisa dibandingkan. Garis putus-putus adalah rata-rata tingkat comeback.
- **Box plot per kategori** untuk faktor yang sama di keempat kategori siswa. Uji Kruskal-Wallis di judul hanya memakai kategori dengan minimal `MIN_N_KELOMPOK` siswa, jadi Comeback Sementara yang kecil hanya tampil sebagai gambaran.

**Cara membaca:** batang dengan n kecil memiliki interval lebar; fokus pada polanya, bukan angka satu batang.

In [ ]:
def tingkat_per_level(x, y, f):
    tipe, y = TIPE[f], np.asarray(y).astype(int)
    if tipe == 'numerik':
        kel = pd.qcut(x.astype(float), q=4, duplicates='drop')
        lab = {iv: (f'{int(np.floor(iv.left)) + 1}' if int(np.floor(iv.left)) + 1 == int(np.floor(iv.right))
                    else f'{int(np.floor(iv.left)) + 1}-{int(np.floor(iv.right))}') for iv in kel.cat.categories}
        level, urutan = kel.map(lab).astype(str).to_numpy(), [lab[iv] for iv in kel.cat.categories]
    elif tipe == 'biner':
        level = np.where(np.asarray(x) == 1, LEVEL_POSITIF[f], LEVEL_NEGATIF[f])
        urutan = [LEVEL_NEGATIF[f], LEVEL_POSITIF[f]]
    elif tipe == 'ordinal':
        level = np.asarray(x).astype(int).astype(str)
        urutan = sorted(set(level), key=int)
    else:
        level = np.asarray(x).astype(str)
        urutan = sorted(set(level))
    rows = []
    for lv in urutan:
        m = level == lv
        k, n = int(y[m].sum()), int(m.sum())
        if n:
            lo, hi = proportion_confint(k, n, method='wilson')
            rows.append(dict(level=lv, n=n, laju=k / n, lo=lo, hi=hi))
    return pd.DataFrame(rows)


TOP = [f for f in HASIL['Matematika']['ringkas'].faktor if TIPE[f] != 'nominal'][:3]
print('Faktor yang ditampilkan:', ', '.join(TOP))

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
for i, (nama, H) in enumerate(HASIL.items()):
    R = H['ringkas'].set_index('faktor')
    for j, f in enumerate(TOP):
        ax = axes[i, j]
        if f not in H['X'].columns:
            ax.set_axis_off()
            continue
        D = tingkat_per_level(H['X'][f], H['y'], f)
        ax.bar(D.level, D.laju, color=WARNA_STATUS[R.loc[f, 'status']], alpha=0.85,
               yerr=[np.clip(D.laju - D.lo, 0, None), np.clip(D.hi - D.laju, 0, None)], capsize=4, ecolor='#555')
        for k_, (v, n) in enumerate(zip(D.laju, D.n)):
            ax.text(k_, 0.02, f'n={n}', ha='center', va='bottom', fontsize=8, color='white' if v > 0.1 else 'black')
        ax.axhline(H['y'].mean(), ls='--', c='k', lw=1)
        ax.set_ylim(0, 1)
        ax.set_title(f'{nama}: {f} (efek {angka(R.loc[f, "efek"])}, skor {R.loc[f, "skor"]}/5)')
        ax.set_xlabel(NAMA[f])
        if j == 0:
            ax.set_ylabel('Proporsi comeback (CI 95%)')
fig.tight_layout()
simpan(fig, 'B6_tingkat_comeback_faktor_kunci')

fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
for i, (nama, H) in enumerate(HASIL.items()):
    f_ = H['fokus']
    urut = [k for k in URUTAN_FOKUS if (f_.kategori == k).any()]
    kat_uji = [k for k in URUTAN_FOKUS if (f_.kategori == k).sum() >= CONFIG['MIN_N_KELOMPOK']]
    for j, f in enumerate(TOP):
        ax = axes[i, j]
        d = pd.DataFrame({'kategori': f_.kategori, 'nilai': f_[f].astype(float) if TIPE[f] != 'biner'
                          else (f_[f] == LEVEL_POSITIF[f]).astype(float)})
        Hs, p = stats.kruskal(*[d.nilai[d.kategori == k] for k in kat_uji])
        if TIPE[f] == 'biner':
            prop = d.groupby('kategori').nilai.mean().reindex(urut)
            ax.bar(range(len(urut)), prop.values, color=[WARNA[k] for k in urut])
            ax.set_ylabel(f'Proporsi {f} = {LEVEL_POSITIF[f]}')
        else:
            sns.boxplot(data=d, x='kategori', y='nilai', order=urut, hue='kategori', hue_order=urut, palette=WARNA,
                        legend=False, showfliers=False, ax=ax)
            sns.stripplot(data=d, x='kategori', y='nilai', order=urut, color='k', alpha=0.3, size=3, ax=ax)
            ax.set_ylabel(f)
        ax.set_xticks(range(len(urut)))
        ax.set_xticklabels(label_kategori(f_, urut), fontsize=8, rotation=15)
        ax.set_xlabel('')
        ax.set_title(f'{nama}: {f} per kategori\nKruskal-Wallis {teks_p("p", p)}')
fig.tight_layout()
simpan(fig, 'B6_faktor_kunci_per_kategori')

## B7. Perbandingan Matematika dan Portugis
**Apa yang dilakukan:** membandingkan ukuran efek setiap faktor di kedua dataset. Perbandingan memakai **ukuran efek, bukan p-value**, karena jumlah siswa di kedua dataset berbeda.

| Kategori | Arti |
|---|---|
| Konsisten | Lolos di kedua dataset dengan arah sama |
| Khas Matematika / Khas Portugis | Lolos hanya di satu dataset |
| Berlawanan | Lolos di keduanya tetapi arahnya berbeda |
| Tidak bermakna | Tidak lolos di keduanya |

**Cara membaca grafik:** titik penuh = lolos, titik kosong = tidak lolos. Garis abu-abu menghubungkan efek faktor yang sama di kedua dataset; garis panjang berarti polanya berbeda antar mata pelajaran.

In [ ]:
Tm = HASIL['Matematika']['ringkas'].set_index('faktor')
Tp = HASIL['Portugis']['ringkas'].set_index('faktor')
rows = []
for f in [f for f in FAKTOR if f in Tm.index or f in Tp.index]:
    em, ep = Tm.efek.get(f, np.nan), Tp.efek.get(f, np.nan)
    lm, lp = Tm.status.get(f) == 'Lolos', Tp.status.get(f) == 'Lolos'
    searah = TIPE[f] == 'nominal' or np.sign(em) == np.sign(ep)
    kat = ('Konsisten' if searah else 'Berlawanan') if (lm and lp) else (
        'Khas Matematika' if lm else ('Khas Portugis' if lp else 'Tidak bermakna'))
    rows.append(dict(faktor=f, tipe=TIPE[f], efek_mat=em, status_mat=Tm.status.get(f, '-'),
                     efek_por=ep, status_por=Tp.status.get(f, '-'), kategori=kat))
BANDING = pd.DataFrame(rows)
urutan_kat = {k: i for i, k in enumerate(['Konsisten', 'Berlawanan', 'Khas Matematika', 'Khas Portugis',
                                          'Tidak bermakna'])}
BANDING = (BANDING.assign(_u=BANDING.kategori.map(urutan_kat), _e=-BANDING.efek_mat.abs())
           .sort_values(['_u', '_e']).drop(columns=['_u', '_e']).reset_index(drop=True))
display(BANDING.round(3))
simpan_tabel(BANDING, 'B7_perbandingan_mat_por')

D = BANDING.assign(urut=BANDING.efek_mat.fillna(0)).sort_values('urut').reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 10.5))
ypos = np.arange(len(D))
ax.hlines(ypos, D[['efek_mat', 'efek_por']].min(axis=1), D[['efek_mat', 'efek_por']].max(axis=1), color='#BDBDBD', lw=2)
for kol, st, nama in [('efek_mat', 'status_mat', 'Matematika'), ('efek_por', 'status_por', 'Portugis')]:
    penuh = (D[st] == 'Lolos').to_numpy()
    ax.scatter(D.loc[penuh, kol], ypos[penuh], color=WARNA[nama], s=60, zorder=3, label=f'{nama} (lolos)')
    ax.scatter(D.loc[~penuh, kol], ypos[~penuh], facecolors='white', edgecolors=WARNA[nama], s=60, zorder=3,
               label=f'{nama} (tidak lolos)')
ax.set_yticks(ypos)
ax.set_yticklabels([f'{f}  [{k}]' for f, k in zip(D.faktor, D.kategori)], fontsize=9)
for v in (-E, E):
    ax.axvline(v, ls='--', c='grey', lw=0.9)
ax.axvline(0, c='k', lw=1)
ax.set_xlabel('Ukuran efek (+ = lebih tinggi / lebih sering pada Comeback)')
ax.set_title('Perbandingan ukuran efek: Matematika vs Portugis')
ax.legend(loc='lower right', fontsize=8)
simpan(fig, 'B7_perbandingan_efek')

## B8. Profil risiko sederhana
**Apa yang dilakukan:** dua faktor non-nominal dengan skor konsistensi tertinggi di Matematika dibagi menjadi "rendah" dan "tinggi" berdasarkan median siswa Matematika dengan G1 < 10, lalu dihitung tingkat comeback di setiap kombinasi. Batas yang sama dipakai untuk Portugis agar bisa dibandingkan.

**Mengapa:** mengubah dua temuan terpisah menjadi satu gambar yang mudah dipakai guru. Batas median hanya berdasarkan sebaran faktor, tidak melihat hasil comeback.

**Cara membaca:** sel dengan n kecil perlu dibaca sebagai pola, bukan angka pasti.

In [ ]:
DUA = TOP[:2]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
PROFIL = {}
for ax, (nama, H) in zip(axes, HASIL.items()):
    b = H['biner']
    kol = {}
    for f in DUA:
        if TIPE[f] == 'biner':
            kol[f] = np.where(b[f] == LEVEL_POSITIF[f], f'{f} = {LEVEL_POSITIF[f]}', f'{f} = {LEVEL_NEGATIF[f]}')
        else:
            med = HASIL['Matematika']['fokus'][f].astype(float).median()
            kol[f] = np.where(b[f].astype(float) <= med, f'rendah (<= {med:g})', f'tinggi (> {med:g})')
    d = pd.DataFrame({DUA[0]: kol[DUA[0]], DUA[1]: kol[DUA[1]], 'comeback': b.comeback.to_numpy()})
    laju = d.pivot_table(index=DUA[0], columns=DUA[1], values='comeback', aggfunc='mean')
    n = d.pivot_table(index=DUA[0], columns=DUA[1], values='comeback', aggfunc='size')
    laju = laju.sort_index().sort_index(axis=1)
    n = n.reindex(index=laju.index, columns=laju.columns).fillna(0)
    PROFIL[nama] = (laju, n)
    annot = np.array([[f'{pct(laju.iloc[r, c])}\n(n = {int(n.iloc[r, c])})' for c in range(laju.shape[1])]
                      for r in range(laju.shape[0])])
    sns.heatmap(laju, annot=annot, fmt='', cmap='Greens', vmin=0, vmax=0.8, cbar=False, ax=ax,
                linewidths=1, linecolor='white', annot_kws={'size': 12, 'weight': 'bold'})
    ax.set_title(f'{nama}: tingkat comeback menurut {DUA[0]} dan {DUA[1]}')
    ax.set_ylabel(DUA[0])
    ax.set_xlabel(DUA[1])
fig.tight_layout()
simpan(fig, 'B8_profil_risiko')

# Diskusi dan kesimpulan
Kesimpulan di bawah disusun langsung dari hasil analisis, sehingga angka selalu sinkron dengan data. Urutannya mengikuti bagian Discussion dan Conclusion pada PPT. Teks juga disimpan sebagai `kesimpulan_otomatis.txt`.

In [ ]:
KATA_ARAH = {'age': ('lebih tua', 'lebih muda'), 'failures': ('lebih banyak', 'lebih sedikit'),
             'traveltime': ('lebih lama', 'lebih singkat')}


ARAH_DESKRIPSI = {'age': ('lebih tua', 'lebih muda'), 'failures': ('lebih banyak', 'lebih sedikit'),
                  'absences': ('lebih banyak', 'lebih sedikit'), 'traveltime': ('lebih lama', 'lebih singkat')}


def deskripsi_faktor(r):
    f, nm = r['faktor'], NAMA[r['faktor']]
    if r['tipe'] in ('ordinal', 'numerik'):
        naik, turun = ARAH_DESKRIPSI.get(f, ('lebih tinggi', 'lebih rendah'))
        return (f'{nm} ({f}) {turun if r["efek"] < 0 else naik} pada siswa Comeback '
                f'(efek {angka(r["efek"])}, {teks_p("q", r["q"])})')
    if r['tipe'] == 'biner':
        return (f'siswa dengan {f} = {LEVEL_POSITIF[f]} {"lebih sering" if r["efek"] > 0 else "lebih jarang"} '
                f'menjadi Comeback (efek {angka(r["efek"])}, {teks_p("q", r["q"])})')
    return f'{nm} ({f}) berbeda distribusinya (V = {angka(r["efek"])}, {teks_p("q", r["q"])})'


def rekomendasi_faktor(r):
    f, nm = r['faktor'], NAMA[r['faktor']]
    if r['tipe'] in ('ordinal', 'numerik'):
        tinggi, rendah = KATA_ARAH.get(f, ('tinggi', 'rendah'))
        teks = f'Siswa gagal G1 dengan {nm} {tinggi if r["efek"] < 0 else rendah} lebih jarang bangkit'
    elif r['tipe'] == 'biner':
        teks = f'Siswa gagal G1 dengan {f} = {LEVEL_NEGATIF[f] if r["efek"] > 0 else LEVEL_POSITIF[f]} lebih jarang bangkit'
    else:
        teks = f'Peluang bangkit berbeda menurut {nm}'
    if r['peran'] == 'dapat diintervensi':
        return teks + f'. Jadikan {nm} indikator pemantauan sejak G1 keluar, lalu uji intervensinya sebagai hipotesis.'
    return teks + f'. Faktor ini tidak bisa diubah sekolah, jadi gunakan hanya untuk mengenali siswa berisiko.'


def susun_kesimpulan():
    L = []
    tulis = L.append
    tulis('KESIMPULAN OTOMATIS')
    tulis('Topik: analisis transisi nilai G1 → G2 → G3, apa yang membedakan siswa comeback (gagal di G1, lulus di G3) '
          'dari siswa yang tetap gagal.')
    tulis('Semua temuan adalah asosiasi dari data observasional, bukan hubungan sebab-akibat.')

    tulis('\n1. Data dan pembersihan')
    for _, r in LOG.iterrows():
        tulis(f'- {r.dataset}: {r.n_awal} siswa, {r.nilai_0_dibuang} dikeluarkan karena nilai 0 (tidak mengikuti '
              f'penilaian), {r.n_akhir} dianalisis. Tidak ada missing value maupun duplikat.')
    tulis(f'- Sekitar {n_irisan} siswa tercatat di kedua dataset, sehingga keduanya dianalisis terpisah.')

    tulis('\n2. EDA dasar: apa yang berkaitan dengan nilai akhir G3')
    for nama, H in HASIL_A.items():
        d = DATA[nama]
        rho_st = stats.spearmanr(d.studytime, d.G3)[0]
        rho_ab = stats.spearmanr(d.absences, d.G3)[0]
        tulis(f'- {nama}: korelasi G1-G3 {angka(H["r_g1"])} dan G2-G3 {angka(H["r_g2"])}. Waktu belajar '
              f'(rho {angka(rho_st)}) dan ketidakhadiran (rho {angka(rho_ab)}) jauh lebih lemah.')
        tulis(f'  Faktor latar belakang terkuat: ' + ', '.join(f'{r.faktor} ({angka(r.rho)})'
                                                              for _, r in H['top_g3'].iterrows()) + '.')
        tulis(f'  Setelah G1 dikontrol, yang tetap bermakna: {daftar(H["bertahan"])}. '
              f'Faktor nominal yang berkaitan dengan G3: {daftar(H["nominal_sig"])}.')
    tulis('  Artinya, titik awal siswa paling menentukan nilai akhir, sehingga analisis comeback lebih berguna untuk '
          'mencari siswa yang masih bisa ditolong.')

    tulis('\n3. Pola transisi nilai (siswa gagal di G1)')
    for nama, H in HASIL.items():
        f_ = H['fokus']
        n, vc = len(f_), f_.kategori.value_counts()
        awal, akhir = int(vc.get('Comeback Awal', 0)), int(vc.get('Comeback Akhir', 0))
        sem, gagal = int(vc.get('Comeback Sementara', 0)), int(vc.get('Tetap Gagal', 0))
        cb = awal + akhir
        porsi = awal / cb if cb else np.nan
        if porsi >= 0.6:
            waktu = f'sebagian besar comeback sudah terjadi di G2 ({pct(porsi)})'
        elif porsi <= 0.4:
            waktu = f'sebagian besar comeback baru terjadi di G3 ({pct(1 - porsi)})'
        else:
            waktu = f'waktu comeback terbagi hampir seimbang ({pct(porsi)} di G2, {pct(1 - porsi)} di G3)'
        tulis(f'- {nama}: dari {n} siswa, {cb} ({pct(cb / n)}) comeback (Awal {awal}, Akhir {akhir}), '
              f'Comeback Sementara {sem}, Tetap Gagal {gagal} ({pct(gagal / n)}); {waktu}.')

    tulis('\n4. G2 sebagai checkpoint')
    for nama, H in HASIL.items():
        g = H['g2']
        tulis(f'- {nama}: siswa gagal G1 yang sudah lulus di G2 berpeluang lulus G3 '
              f'{pct(g.loc["Lulus G2", "peluang lulus G3"])} (n = {g.loc["Lulus G2", "n"]}), sedangkan yang masih '
              f'gagal di G2 hanya {pct(g.loc["Gagal G2", "peluang lulus G3"])} (n = {g.loc["Gagal G2", "n"]}); '
              f'uji Fisher {teks_p("p", H["g2_p"])}.')

    tulis('\n5. Faktor yang membedakan Comeback dan Tetap Gagal (skor konsistensi 0 sampai 5)')
    for nama, H in HASIL.items():
        R = H['ringkas']
        kunci = R[R.status == 'Lolos']
        if len(kunci):
            tulis(f'- {nama}: {len(kunci)} faktor lolos.')
            for i, (_, r) in enumerate(kunci.iterrows(), 1):
                cat = f' [{r.catatan}]' if r.catatan else ''
                tulis(f'   {i}) {deskripsi_faktor(r)}; skor {r.skor}/5{cat}.')
        else:
            top = R.nlargest(3, 'efek_abs')
            tulis(f'- {nama}: tidak ada faktor yang lolos. Efek terbesar (eksploratif): '
                  + '; '.join(deskripsi_faktor(r) for _, r in top.iterrows()) + '.')
        tulis('   Peringkat signifikansi (q terkecil): '
              + ', '.join(f'{r.faktor}' for _, r in R.sort_values(['q', 'p']).head(5).iterrows()) + '.')
        tulis('   Peringkat kekuatan (|efek| terbesar): '
              + ', '.join(f'{r.faktor} ({angka(r.efek)})' for _, r in R.nlargest(5, 'efek_abs').iterrows()) + '.')
        st = R.set_index('faktor').status
        umum = [f for f in ['studytime', 'paid', 'schoolsup', 'famsup', 'internet', 'higher']
                if f in st.index and st[f] == 'Tidak bermakna']
        if umum:
            tulis(f'   Faktor yang sering dianggap penting tetapi tidak membedakan: {daftar(umum)}.')
        if H['pasangan_kuat']:
            tulis('   Faktor kunci yang saling berkorelasi kuat (dibaca sebagai satu temuan): '
                  + ', '.join(f'{a_} & {b_}' for a_, b_, _ in H['pasangan_kuat']) + '.')

    tulis('\n6. Perbandingan Matematika dan Portugis')
    for k in ['Konsisten', 'Berlawanan', 'Khas Matematika', 'Khas Portugis']:
        tulis(f'- {k}: {daftar(BANDING[BANDING.kategori == k].faktor.tolist())}.')
    khas = BANDING[BANDING.kategori.isin(['Khas Matematika', 'Khas Portugis'])]
    searah = [r.faktor for _, r in khas.iterrows() if r.tipe != 'nominal'
              and np.sign(r.efek_mat) == np.sign(r.efek_por)
              and min(abs(r.efek_mat), abs(r.efek_por)) >= CONFIG['EFEK_REPLIKASI']]
    if searah:
        tulis(f'- Searah di kedua dataset tetapi hanya lolos di satu dataset (pola sama, kekuatan berbeda): '
              f'{daftar(searah)}.')

    tulis('\n7. Profil risiko sederhana')
    for nama, (laju, n) in PROFIL.items():
        s = laju.stack().dropna()
        hi_, lo_ = s.idxmax(), s.idxmin()
        tulis(f'- {nama}: comeback tertinggi pada {DUA[0]} {hi_[0]} dan {DUA[1]} {hi_[1]} ({pct(s.max())}, '
              f'n = {int(n.loc[hi_])}); terendah pada {DUA[0]} {lo_[0]} dan {DUA[1]} {lo_[1]} ({pct(s.min())}, '
              f'n = {int(n.loc[lo_])}).')

    tulis('\n8. Jawaban atas tujuan penelitian')
    km = HASIL['Matematika']['ringkas']
    kunci_m = km[km.status == 'Lolos'].faktor.tolist()
    tulis(f'- Tujuan 1: data bersih berisi {LOG.n_akhir.iloc[0]} siswa Matematika dan {LOG.n_akhir.iloc[1]} siswa '
          'Portugis.')
    tulis('- Tujuan 2 dan 3: distribusi dan hubungan antar variabel divisualisasikan dengan histogram, scatter plot, '
          'bar chart, box plot, dan heatmap korelasi (Bagian A).')
    tulis(f'- Tujuan 4: nilai sebelumnya (G1, G2) adalah penentu terkuat G3; faktor latar belakang terkuat di '
          f'Matematika adalah {HASIL_A["Matematika"]["top_g3"].faktor.iloc[0]}.')
    tulis(f'- Tujuan 5: di Matematika, faktor yang membedakan siswa comeback adalah {daftar(kunci_m)}.')

    tulis('\n9. Keterbatasan')
    tulis('- Data survei satu waktu dari dua sekolah di Portugal (2005/2006): temuan bersifat asosiasi dan '
          'generalisasinya terbatas.')
    tulis('- Kelompok kecil: ' + '; '.join(f'{n} {int(H["y"].sum())} Comeback vs {int((H["y"] == 0).sum())} '
                                           f'Tetap Gagal' for n, H in HASIL.items()) + '.')
    tulis(f'- {len(HASIL["Matematika"]["aktif"])} faktor diuji sekaligus; koreksi BH mengurangi tetapi tidak '
          'menghilangkan temuan palsu. Faktor bertanda "sensitif terhadap metode koreksi" tidak lolos koreksi Holm '
          'yang lebih ketat.')
    tulis('- goout, Dalc, Walc, studytime, dan freetime dilaporkan sendiri oleh siswa, sehingga rentan bias.')
    tulis('- Siswa dengan nilai 0 dikeluarkan, padahal mereka mungkin justru kelompok yang paling butuh bantuan.')

    tulis('\n10. Rekomendasi (hipotesis tindakan, bukan klaim sebab-akibat)')
    ada = False
    for nama, H in HASIL.items():
        R = H['ringkas']
        for _, r in R[(R.status == 'Lolos') & (R.skor >= 4)].head(3).iterrows():
            tulis(f'- [{nama}] {rekomendasi_faktor(r)}')
            ada = True
    if not ada:
        tulis('- Belum ada faktor non-nilai yang cukup konsisten untuk dasar intervensi. Fokuskan pemantauan pada G1 '
              'dan G2.')
    tulis('- Pakai G2 sebagai checkpoint kedua: siswa yang masih gagal di G2 perlu pendampingan sebelum ujian akhir.')
    tulis('- Penelitian lanjutan dapat menguji intervensi pada faktor yang dapat diubah dengan kelompok pembanding.')

    teks = '\n'.join(L)
    with open(os.path.join(OUT, 'kesimpulan_otomatis.txt'), 'w', encoding='utf-8') as fh:
        fh.write(teks)
    return teks


print(susun_kesimpulan())

## Peta slide presentasi (20 menit)

| No | Slide | Isi utama | Grafik atau tabel | Menit |
|---|---|---|---|---|
| 1 | Judul dan anggota | Judul, nama 3 anggota | - | 0,5 |
| 2 | Latar belakang dan motivasi | Kelulusan ditentukan sepanjang tahun | - | 1,5 |
| 3 | Tantangan dan ide | G1 hampir menentukan G3, maka fokus ke siswa comeback | `A3_scatter_nilai_sebelumnya` (panel zona comeback) | 1,5 |
| 4 | Tujuan penelitian | Lima tujuan | - | 1 |
| 5 | Data | Dua dataset, 30 faktor, empat tipe data | Tabel kamus ringkas | 1 |
| 6 | Pembersihan data | Siswa nilai 0 dan alasannya | `A1_g3_sebelum_cleaning`, log cleaning | 1,5 |
| 7 | EDA: distribusi dan nilai sebelumnya | Histogram, scatter G1/G2 vs G3 | `A2_distribusi_nilai`, `A3_scatter_nilai_sebelumnya` | 2 |
| 8 | EDA: waktu belajar, ketidakhadiran, korelasi | Bar, box plot, heatmap | `A4_studytime_failures_absences`, `A5_heatmap_korelasi` | 2 |
| 9 | EDA: faktor vs nilai akhir | Sebelum dan sesudah G1 dikontrol | `A6_faktor_vs_g3` | 1,5 |
| 10 | Pola transisi | Siapa yang bangkit dan kapan, checkpoint G2 | `B2_sankey`, `B2_komposisi_dan_lintasan`, `B3_checkpoint_g2` | 2 |
| 11 | Faktor pembeda comeback | 30 faktor dengan ukuran efek | `B4_forest_plot_faktor` | 2 |
| 12 | Seberapa kuat temuannya | Matriks konsistensi dan perbandingan Portugis | `B5_matriks_konsistensi`, `B7_perbandingan_efek` | 1,5 |
| 13 | Profil risiko | Faktor kunci per level dan kombinasi | `B6_tingkat_comeback_faktor_kunci`, `B8_profil_risiko` | 1 |
| 14 | Kesimpulan, keterbatasan, rekomendasi | Poin 8 sampai 10 kesimpulan otomatis | - | 1 |
| | **Total** | | | **20** |

Grafik `B6_faktor_kunci_per_kategori` dan tabel lengkap disimpan sebagai cadangan untuk sesi tanya jawab.

## Ekspor hasil
Semua grafik, tabel, dan kesimpulan dikemas menjadi `hasil_analisis.zip`. Di Colab file otomatis terunduh.

In [ ]:
arsip = shutil.make_archive('hasil_analisis', 'zip', OUT)
print(f'{len(os.listdir(OUT))} file dikemas ke {arsip}')
try:
    from google.colab import files
    files.download(arsip)
except ImportError:
    pass